# Computing and Visualizing KPIs using Python

In [ ]:
# Loading data
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# metadata 
print(bank_marketing.metadata) 
  
# variable information 
print(bank_marketing.variables) 


In [ ]:
# Consstruct dataframe from X,y
import pandas as pd
df = pd.DataFrame(X)
df['y'] = y
df.head()

In [ ]:
# Create a conversion data column
df['conversion'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)
df.head()

## Aggreate converrtion rate

In [ ]:
print(f"Total Conversions: {df.conversion.sum()} out of {df.shape[0]} clients")

In [ ]:
print(f"The conversion rate is: {df.conversion.mean():.2%}")

## Conversion rate by age

In [ ]:
conversion_rate_by_age = df.groupby('age')['conversion'].mean()*100
conversion_rate_by_age

In [ ]:
# Visualize conversion rate by age using plotly
import plotly.express as px
figsize=(10,7)
fig = px.line(conversion_rate_by_age, 
              x=conversion_rate_by_age.index, 
              y=conversion_rate_by_age.values, 
              title='Conversion Rate by Age', 
              labels={'x':'Age', 'y':'Conversion Rate (%)'},
              template='ggplot2'
              )
fig.show()

In [ ]:
# To group bank clients into six different groups, based on their age—between 18 and 30, between 30 and 40, between 40 and 50, between 50 and 60, between 60 and 70, and 70 and older
age_bins = [18, 30, 40, 50, 60, 70, float('inf')]
age_labels = ['18-30', '30-40', '40-50', '50-60', '60-70', '70+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)
df.head()

In [ ]:
# Create conversion rate by age group
conversion_rate_by_age_group = df.groupby('age_group')['conversion'].mean()*100
conversion_rate_by_age_group

In [ ]:
# Visualize conversion rate by age group using plotly
from plotly import colors


figsize=(10,7)
fig = px.bar(conversion_rate_by_age_group,
            x=conversion_rate_by_age_group.index, 
            y=conversion_rate_by_age_group.values, 
            title='Conversion Rate by Age Group', 
            labels={'x':'Age Group', 'y':'Conversion Rate (%)'},
            template='ggplot2',
            color_discrete_sequence=colors.qualitative.Pastel
            )
fig.show()

## Non conversions vs conversions

In [ ]:
df['marital'].value_counts(dropna=False)

In [ ]:
pd.pivot_table(df,values='y', index='marital', columns='conversion', aggfunc=len)

In [ ]:
# Create a pie chart to visualize the distribution of marital status among converters and non-converters
conversion_marital_counts = df.groupby(['marital', 'conversion']).size().reset_index(name='count')
fig = px.pie(conversion_marital_counts,
             names='marital',
             values='count',
             title='Marital Status Distribution among Converters and Non-Converters',
             template='ggplot2',
             color_discrete_sequence=colors.qualitative.Pastel
             )
fig.show()

In [ ]:
marital_by_conversion = df.groupby(['conversion', 'marital']).size().unstack()

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
                    subplot_titles=['Non-Conversion (0)', 'Conversion (1)'])

for i, conversion in enumerate(marital_by_conversion.index):
    fig.add_trace(
        go.Pie(
            labels=marital_by_conversion.columns,
            values=marital_by_conversion.loc[conversion],
            name=str(conversion),
            textinfo='percent+label',
            marker=dict(colors=colors.qualitative.Pastel)
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title='Marital Status Distribution by Conversion',
    template='ggplot2'
)
fig.show()


# Conversion rate by age and marital status

In [ ]:
df.groupby(['age_group', 'marital'])['conversion'].sum()

In [ ]:
df.groupby(['age_group', 'marital'])['conversion'].sum().unstack('marital').fillna(0)

In [ ]:
age_marital_df = df.groupby(['age_group', 'marital'])['conversion'].sum().unstack('marital').fillna(0)
age_marital_df

In [ ]:
age_marital_df = age_marital_df.divide(df.groupby(by='age_group')['conversion'].count(), axis=0)
age_marital_df

In [ ]:
plot_df = age_marital_df.loc[age_labels].mul(100).reset_index()

fig = px.bar(
    plot_df,
    x='age_group',
    y=['divorced', 'married', 'single'],
    barmode='group',
    title='Conversion rates by Age & Marital Status',
    labels={
        'age_group': 'age group',
        'value': 'conversion rate (%)',
        'variable': 'marital status'
    },
    template='ggplot2',
    color_discrete_sequence=colors.qualitative.Pastel
)

fig.show()




In [ ]:
fig = px.bar(
    age_marital_df.loc[age_labels].mul(100).reset_index(),
    x='age_group',
    y=['divorced', 'married', 'single'],
    barmode='stack',
    title='Conversion rates by Age & Marital Status',
    labels={
        'age_group': 'age group',
        'value': 'conversion rate (%)',
        'variable': 'marital status'
    },
    template='ggplot2',
    color_discrete_sequence=colors.qualitative.Pastel
)

fig.show()
